In [1]:
import polars as pl
import polars.selectors as cs
import plotly.express as px

In [2]:
df = pl.read_parquet("../data/processed/clean_tripdata.parquet")
df.glimpse()

Rows: 6006904
Columns: 26
$ ride_id                       <str> 'B7516B7B2708CC24', 'BCF70C0B42130B92', 'E004845CDF26C9D4', 'CE6ACBC2A319AA59', '12F3FE6984F93383', '6B3160CE448781BA', 'FB10CC36BED3C86B', '4D47190D0D5E6E55', '02531D76F32E6F6B', '20E8890ED8FCBF1B'
$ rideable_type                 <str> 'classic_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'electric_bike', 'electric_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'electric_bike'
$ started_at           <datetime[μs]> 2025-10-05 15:26:37.782000, 2026-09-11 02:06:11.044000, 2025-10-14 17:05:58.193000, 2026-05-27 10:27:11.879000, 2026-08-24 05:37:01.378000, 2025-10-04 11:53:40.309000, 2026-05-18 12:49:36.303000, 2026-05-24 12:27:14.893000, 2026-06-02 14:15:13.902000, 2026-07-26 15:41:55.611000
$ ended_at             <datetime[μs]> 2025-10-05 15:29:47.109000, 2026-09-11 02:10:50.196000, 2025-10-14 17:14:50.362000, 2026-05-27 10:29:47.582000, 2026-08-24 05:40:45.344000, 2025-10-04 12:15:08.315000, 2026-05-18 1

In [3]:
df.select(cs.numeric()).describe()

statistic,start_lat,start_lng,end_lat,end_lng,ride_length,day_of_week,start_hour,end_hour,month,distance_km
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",41.904664,-87.646991,41.905052,-87.647359,855.390892,4.009267,14.040382,14.182858,6.993687,2.247817
"""std""",0.044228,0.0269,0.044428,0.027177,1612.84099,1.949833,5.034889,5.086577,2.59172,1.975599
"""min""",41.64,-87.91,41.51,-88.1,60.0,1.0,0.0,0.0,1.0,0.0
"""25%""",41.88281,-87.66,41.882906,-87.660141,334.0,2.0,10.0,11.0,5.0,0.969966
"""50%""",41.9,-87.6429,41.9,-87.643353,566.0,4.0,15.0,15.0,7.0,1.654917
"""75%""",41.93,-87.63,41.930585,-87.63,977.0,6.0,18.0,18.0,9.0,2.91763
"""max""",42.07,-87.52,42.19,-87.37,86398.0,7.0,23.0,23.0,12.0,48.370801


In [ ]:
# seconds -> minutes
df = df.with_columns(pl.col("ride_length").truediv(60).round(2))

# Station columns used for the null-rate metrics
STATION_COLS = ["start_station_name", "end_station_name"]

In [ ]:
df["ride_length"].quantile([.75, .9, .99, .999])

In [ ]:
df["ride_length"].std() / df["ride_length"].mean()

In [ ]:
fig = px.histogram(
    df,
    x="ride_length",
    nbins=60,
    labels={"ride_length": "Ride Length in Minutes"},
)
fig.show()

In [ ]:
def profile(d: pl.DataFrame) -> pl.DataFrame:
    """Per-column summary: dtype, nulls (count and %), and unique values."""
    nulls = d.null_count().row(0)
    return pl.DataFrame({
        "column": d.columns,
        "dtype": [str(t) for t in d.dtypes],
        "nulls": nulls,
        "pct_null": [round(n / d.height * 100, 1) for n in nulls],
        "uniques": [d[c].n_unique() for c in d.columns],
    })


# Available statistics for continuous metrics (name -> expression builder)
STATS = {
    "avg": lambda c: c.mean(),
    "med": lambda c: c.median(),
    "std": lambda c: c.std(),
    "min": lambda c: c.min(),
    "max": lambda c: c.max(),
    "p90": lambda c: c.quantile(0.90),
    "p99": lambda c: c.quantile(0.99),
}


def summarize_by(
    d: pl.DataFrame,
    by: str,
    metrics: tuple[str, ...] = ("ride_length", "distance_km"),
    stats: tuple[str, ...] = ("avg", "med", "std"),
    share: bool = False,
    station_nulls: bool = False,
    digits: int = 2,
) -> pl.DataFrame:
    """Statistics of CONTINUOUS metrics (e.g. ride_length), grouped by `by`.

    stats: any keys of STATS.
    share: adds the group's % of total rides.
    station_nulls: adds the % of null stations WITHIN each group.
    """
    aggs = []
    if share:
        aggs.append(pl.len().mul(100 / d.height).alias("rides_pct"))
    for m in metrics:
        aggs += [STATS[s](pl.col(m)).round(digits).alias(f"{m}_{s}") for s in stats]
    if station_nulls:
        aggs += [
            pl.col(c).is_null().mean().mul(100).round(digits).alias(f"{c}_null_pct")
            for c in STATION_COLS
        ]
    return d.group_by(by).agg(aggs).sort(by)


def distribution_by(
    d: pl.DataFrame,
    by: str,
    col: str,
    pivot: bool = True,
    digits: int = 2,
) -> pl.DataFrame:
    """% of rides of each `by` group falling in each value of `col`.

    Each `by` group sums to 100%, so groups of different sizes are comparable.
    pivot=True: one row per `col` value, one column per `by` group.
    """
    out = (
        d.group_by(by, col)
        .agg(pl.len().alias("rides"))
        .with_columns(
            pl.col("rides").truediv(pl.col("rides").sum().over(by)).mul(100).round(digits).alias("pct")
        )
        .sort(col, by)
    )
    if pivot:
        out = out.pivot(on=by, index=col, values="pct").sort(col)
    return out


profile(df)

In [ ]:
summarize_by(df, "member_casual", share=True, station_nulls=True)

In [ ]:
summarize_by(df, "rideable_type", share=True, station_nulls=True)

In [ ]:
summarize_by(df, "day_of_week", share=True, station_nulls=True)

In [ ]:
summarize_by(df, "season", share=True, station_nulls=True)

In [ ]:
summarize_by(df, "month", share=True, station_nulls=True)

In [ ]:
for col in ("start_hour", "day_of_week", "month", "season"):
    print(col)
    display(distribution_by(df, "member_casual", col))

In [ ]:
(
    df.filter(pl.col("is_round_trip").is_not_null())
    .group_by("member_casual", "is_round_trip")
    .agg(pl.len().alias("total_rides"))
    .with_columns(
        (
            pl.col("total_rides")
            / pl.col("total_rides").sum().over("member_casual")
            * 100
        ).round(2).alias("percentage")
    )
    .sort("total_rides", descending=True)
)

In [ ]:
(
    df.select(STATION_COLS)
    .unpivot(value_name="station")
    .filter(pl.col("station").is_not_null())
    .group_by("station")
    .agg(pl.len().alias("total_rides"))
    .top_k(10, by="total_rides")
    .sort("total_rides", descending=True)
)